<h3 style="color:#5A4CE1;font-weight:600;">1 | Clean your Data</h3>

We use data from 80,000 UFO sightings, gathered by NUFORC (The National UFO Reporting Center). This data has some interesting descriptions of UFO sightings, for example:

- Long example description
- Short example description

The ufos.csv spreadsheet includes columns about the `city`, `state` and `country` where the sighting occurred, the object's `shape` and its `latitude` and `longitude`.

In [60]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

ufos = pd.read_csv("./data/ufos.csv")
ufos.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


Create a small dataframe from the above with new titles. Also, check the unique values in te `Country` field.

In [61]:
ufos = pd.DataFrame(
    {
        "Seconds": ufos["duration (seconds)"],
        "Country": ufos["country"],
        "Latitude": ufos["latitude"],
        "Longitude": ufos["longitude"],
    }
)

ufos.Country.unique()

<StringArray>
['us', nan, 'gb', 'ca', 'au', 'de']
Length: 6, dtype: str

Since the dataset contains 80,332 rows, we can reduce the amount of data to deal with by:
- dropping any null values and
- only dealing with sightings between 1-60 secs.

In [62]:
ufos.dropna(inplace=True)
ufos = ufos[(ufos.Seconds <= 60) & (ufos.Seconds >= 1)]

ufos.info()

<class 'pandas.DataFrame'>
Index: 25863 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    25863 non-null  float64
 1   Country    25863 non-null  str    
 2   Latitude   25863 non-null  float64
 3   Longitude  25863 non-null  float64
dtypes: float64(3), str(1)
memory usage: 1010.3 KB


Convert the text values for countries to a number e.g. if there are 5 unique countries, they'll be converted to values ranging from 0-4

In [63]:
le = LabelEncoder()
ufos.Country = le.fit_transform(ufos.Country)
ufos.head()

,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
23,60.0,4,45.582778,-122.352222
24,3.0,3,51.783333,-0.783333


<h3 style="color:#5A4CE1;font-weight:600;">2 | Build your Model</h3>

The X vector contains the features `Seconds`, `Latitude` and `Longitude` with which to train the model.<br/>
The y vector, the `Country` columns is the label we want to predict.

In [64]:
selected_features = ["Seconds", "Latitude", "Longitude"]
X = ufos[selected_features]
y = ufos["Country"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

Train the model using **Logistic Regression**.

In [65]:
model = LogisticRegression(solver="newton-cholesky")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(classification_report(y_test, predictions))
print("Predicted labels: ", predictions)
print(f"Accuracy: {accuracy_score(y_test,predictions):.2f}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.85      0.47      0.60       250
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00       131
           4       0.97      1.00      0.98      4743

    accuracy                           0.97      5173
   macro avg       0.96      0.89      0.92      5173
weighted avg       0.97      0.97      0.97      5173

Predicted labels:  [4 4 4 ... 3 4 4]
Accuracy: 0.97


<h3 style="color:#5A4CE1;font-weight:600;">3 | 'Pickle' your Model</h3>

'Pickle' (preserve on disk) the model, and load it to test it against a sample data array with values for seconds, latitude and longitude.

In [66]:
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names")
# NOTE: Since we fitted the model using a DataFrame, and not an ndarray, if we don't
#       filter the warnings, a warning will appear when we test the model with a simple
#       ndarray such as [50, 44, -12]. The solution is to either pass a DataFrame with
#       named columns,

model_filename = "ufo-model.pkl"
pickle.dump(model, open(model_filename, "wb"))

model = pickle.load(open(model_filename, "rb"))
print(model.predict([[50, 44, -12]]))

[3]


To be able to convert the numeric prediction for a country, the 3 above, to an actual country name, we could **save/load the encoder together with the model**:

In [67]:
# Save both model and LabelEncoder
model_le_filename = "ufo-model-encoder.pkl"
with open(model_le_filename, "wb") as f:
    pickle.dump(
        {
            "model": model,
            "label_encoder": le,
            "features": selected_features,
        },
        f,
    )

# Load and predict
with open(model_le_filename, "rb") as f:
    data = pickle.load(f)

loaded_model = data["model"]
loaded_le = data["label_encoder"]
features = data["features"]

sample = [[50, 44, -12]]
pred_num = loaded_model.predict(sample)
pred_label = loaded_le.inverse_transform(pred_num)
print(f"Numeric prediction: {pred_num}")
print(f"Country prediction: {pred_label}")

Numeric prediction: [3]
Country prediction: ['gb']


<h3 style="color:#5A4CE1;font-weight:600;">4 | Build a Flask app</h3>
see the <a href="README.md#flask" alt="Flask Web-app">README</a> file